### Train a bilstm to generate jokes in forward and reverse based on a controlled bit

In [ ]:
import pickle
import importlib
import random
from time import time

import numpy as np

import keras
from keras.preprocessing.text import Tokenizer
from keras.utils import to_categorical
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential, Input
from keras.layers import Dense, Dropout, Activation
from keras.layers import LSTM, Bidirectional
from keras.layers import Embedding, TimeDistributed, Flatten, Merge, Concatenate
from keras import regularizers
from keras.metrics import sparse_categorical_accuracy, sparse_categorical_crossentropy
from keras.models import load_model
from keras.optimizers import Adam
from keras.models import Model

from keras.callbacks import TensorBoard, EarlyStopping, ModelCheckpoint
import tensorflow as tf
from keras import backend as K

### Variables

In [ ]:
# http://nlp.stanford.edu/data/glove.6B.zip
DATA_PATH = '../datasets/jokes.pickle'
VOCAB_PATH = '../datasets/jokes_vocabulary.pickle'
MODELS_PATH = '../models/'

GLOVE_PATH = '../data/glove.6B.200d.txt'

MODEL_PREFIX = 'jokes_controlled_stacked_lstm_glove'

MAX_SEQUENCE_LENGTH = 13
VALIDATION_SPLIT = 0.2

GLOVE_EMBEDDING_DIM = 200
EMBEDDING_DIM1 = 256
EMBEDDING_DIM2 = 256
HIDDEN_DIM1 = 512
HIDDEN_DIM2 = 256
DEEPER_DIM = 256
DROPOUT_FACTOR = 0.2
REGULARIZATION = 0.00001
LEARNING_RATE = 0.003

DATA_PERCENT = 0.1

RUN_INDEX = 1

In [ ]:
with open(DATA_PATH, 'rb') as pickleFile:
    sentences = pickle.load(pickleFile)

with open(VOCAB_PATH, 'rb') as pickleFile:
    vocab = pickle.load(pickleFile)
    
random.shuffle(sentences)

print("Number of sentences = ", len(sentences))
print(sentences[:2])
print("Vocab size = ", len(vocab))
print(vocab[:10])

In [ ]:
# tokenize data
num_words = len(vocab)

tokenizer = Tokenizer(num_words=None, filters='', lower=True, split=' ', 
                      char_level=False, oov_token=None)
tokenizer.fit_on_texts(sentences)
assert num_words == len(tokenizer.word_index)

encoded_sentences = tokenizer.texts_to_sequences(sentences)
print(encoded_sentences[:5])

VOCAB_SIZE = len(tokenizer.word_index) + 1
print(VOCAB_SIZE)

In [ ]:
# saving
with open(MODELS_PATH + MODEL_PREFIX + '_tokenizer_' + str(RUN_INDEX) + '.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

### Preparing Training Data

In [ ]:
TAG_SIZE = 2
X_data = []
y_data = []
for sentence in encoded_sentences:
    l = len(sentence)
    sliding_window_length = min(l-3, MAX_SEQUENCE_LENGTH)
    step_size = 1
    for i in range(0, l - sliding_window_length, step_size):
        X_data.append(sentence[i:i+sliding_window_length])
        y_data.append(sentence[i+1:i+sliding_window_length+1])
        
print("Total training data size = ", len(X_data))
MAX_SEQ_LEN = max([len(seq) for seq in X_data])
print("Max seq len = ", MAX_SEQ_LEN)

forward_X_data = pad_sequences(X_data, maxlen=MAX_SEQ_LEN, padding='pre')
forward_y_data = pad_sequences(y_data, maxlen=MAX_SEQ_LEN, padding='pre').reshape(-1, MAX_SEQ_LEN, 1)
forward_tag = to_categorical(np.full((forward_X_data.shape[0], MAX_SEQ_LEN), 0), TAG_SIZE)

#print(forward_X_data.shape)
#print(forward_X_data[:2])
#print(forward_y_data.shape)
#print(forward_y_data[:2])
#print(forward_tag.shape)

reverse_X_data = pad_sequences([item[::-1] for item in y_data], maxlen=MAX_SEQ_LEN, padding='pre')
reverse_y_data = pad_sequences([item[::-1] for item in X_data], maxlen=MAX_SEQ_LEN, padding='pre').reshape(-1, MAX_SEQ_LEN, 1)
reverse_tag = to_categorical(np.full((reverse_X_data.shape[0], MAX_SEQ_LEN), 1), TAG_SIZE)

#print(reverse_X_data.shape)
#print(reverse_X_data[:2])
#print(reverse_y_data.shape)
#print(reverse_y_data[:2])
#print(reverse_tag.shape)

X_data = np.concatenate((forward_X_data, reverse_X_data), axis=0)
y_data = np.concatenate((forward_y_data, reverse_y_data), axis=0)
tag_data = np.concatenate((forward_tag, reverse_tag), axis=0)

# shuffle
perm = np.random.permutation(X_data.shape[0])
X_data = X_data[perm]
y_data = y_data[perm]
tag_data = tag_data[perm]
print(X_data.shape)
print(X_data[:4])
print(y_data.shape)
print(y_data[:4])
print(tag_data.shape)
print(tag_data[:4])

In [ ]:
print('Indexing glove word vectors')
#Glove Vectors
glove_embeddings_index = {}
f = open(GLOVE_PATH)
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    glove_embeddings_index[word] = coefs
f.close()
print('Total %s word vectors.' % len(glove_embeddings_index))

In [ ]:
print('Preparing glove embedding matrix')
glove_embedding_matrix = np.zeros((VOCAB_SIZE, GLOVE_EMBEDDING_DIM))
for word,i in tokenizer.word_index.items():
    embedding_vector = glove_embeddings_index.get(word)
    if embedding_vector is not None:
        # words not found in embedding index will be all-zeros.
        glove_embedding_matrix[i] = embedding_vector
print('Null word embeddings: %d' % np.sum(np.sum(glove_embedding_matrix, axis=1) == 0))
print(glove_embedding_matrix.shape)

In [ ]:
# define model
def StackedLSTM(vocab_size, glove_embedding_dim, glove_embedding_matrix, embedding_dim1, embedding_dim2,
           hidden_dim1, hidden_dim2, deeper_dim, max_seq_len, tag_size,
           dropout_factor=0.5, regularization=0.00001, learning_rate=0.001):
    
    inputs = Input(shape=(None,))
    tag_inputs = Input(shape=(None,tag_size))
    
    glove_embedding = Embedding(vocab_size, glove_embedding_dim, #input_length=max_seq_len,
                                  weights=[glove_embedding_matrix],
                                  mask_zero=True,trainable=False)(inputs)
    
    word_embedding = Embedding(vocab_size, embedding_dim1, #input_length=max_seq_len, 
                               mask_zero=True, embeddings_regularizer=regularizers.l2(regularization))(inputs)
    
    #tag_embedding = Embedding(tag_size, tag_size, embeddings_regularizer=regularizers.l2(regularization))(tag_inputs)
    
    concat_embeds = Concatenate(axis=-1)([glove_embedding, word_embedding, tag_inputs])
    
    final_embed = Dense(units=embedding_dim2, activation='tanh',
                        kernel_regularizer=regularizers.l2(regularization))(concat_embeds)
    
    lstm1 = LSTM(hidden_dim1, activation='tanh', 
                   kernel_regularizer=regularizers.l2(regularization), 
                   recurrent_regularizer=regularizers.l2(regularization), #unroll=True, 
                   return_sequences = True, dropout=dropout_factor, recurrent_dropout=dropout_factor)(final_embed)
    
    lstm2 = LSTM(hidden_dim2, activation='tanh', 
                   kernel_regularizer=regularizers.l2(regularization), 
                   recurrent_regularizer=regularizers.l2(regularization), #unroll=True, 
                   return_sequences = True, dropout=dropout_factor, recurrent_dropout=dropout_factor)(lstm1)
    
    timedist_dropout = TimeDistributed(Dropout(dropout_factor))(lstm2)
    
    deep_dense = Dense(units=deeper_dim, activation='tanh', 
                       kernel_regularizer=regularizers.l2(regularization))(timedist_dropout)
    
    dropout_layer1 = Dropout(dropout_factor)(deep_dense)
    
    outputs = Dense(units=vocab_size, activation='softmax', 
                    kernel_regularizer=regularizers.l2(regularization))(dropout_layer1)
    
    model = Model(inputs=[inputs,tag_inputs], outputs=outputs)
    
    model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(lr=learning_rate),
                  metrics=[sparse_categorical_crossentropy, sparse_categorical_accuracy], sample_weight_mode='temporal')
    return model

In [ ]:
K.clear_session()
sess = tf.Session()
K.set_session(sess)

model = StackedLSTM(vocab_size=VOCAB_SIZE, glove_embedding_dim=GLOVE_EMBEDDING_DIM,
                    glove_embedding_matrix=glove_embedding_matrix, 
                    embedding_dim1=EMBEDDING_DIM1, embedding_dim2=EMBEDDING_DIM2,
                    hidden_dim1=HIDDEN_DIM1, hidden_dim2=HIDDEN_DIM2,
                    deeper_dim=DEEPER_DIM, max_seq_len=MAX_SEQ_LEN, dropout_factor=DROPOUT_FACTOR, 
                    regularization=REGULARIZATION, learning_rate=LEARNING_RATE, tag_size=TAG_SIZE)
print(model.summary())

In [ ]:
class TB(TensorBoard):
    def __init__(self, log_every=1, **kwargs):
        super().__init__(**kwargs)
        self.log_every = log_every
        self.counter = 0
    
    def on_batch_end(self, batch, logs=None):
        self.counter+=1
        if self.counter%self.log_every==0:
            for name, value in logs.items():
                if name in ['batch', 'size']:
                    continue
                summary = tf.Summary()
                summary_value = summary.value.add()
                summary_value.simple_value = value.item()
                summary_value.tag = name
                self.writer.add_summary(summary, self.counter)
            self.writer.flush()
        
        super().on_batch_end(batch, logs)

In [ ]:
start_time = time()
tensorboard = TB(log_dir="./logs/" + MODEL_PREFIX + "/{}".format(time()), 
                          histogram_freq=0, write_graph=True, write_images=False, log_every=10)

callbacks=[tensorboard, 
           EarlyStopping(patience=5, monitor='val_loss'),
           ModelCheckpoint(filepath=MODELS_PATH + 'checkpoints/'+ MODEL_PREFIX + '_gen' + str(RUN_INDEX) + '.{epoch:02d}-{val_loss:.2f}.hdf5', 
                           monitor='val_loss', verbose=1, mode='auto', period=1), 
           ModelCheckpoint(filepath=MODELS_PATH + MODEL_PREFIX + '_gen'+str(RUN_INDEX)+'.hdf5', 
                           monitor='val_loss', verbose=1, mode='auto', period=1, save_best_only=True)]

model.fit([X_data, tag_data], y_data, epochs=10, batch_size=1024, shuffle=True, verbose=1, validation_split=0.2, callbacks=callbacks)

print("Total elapsed time: ", time()-start_time)

In [ ]:
# generate a sequence from a language model
def generate_categorical(model, tokenizer, seed_text, maxlen, probabilistic=False, exploration_factor=1.0, tag=0):
    
    reverse_word_map = dict(map(reversed, tokenizer.word_index.items()))
    seq = tokenizer.texts_to_sequences([seed_text])[0]
    print(seq)
    
    while True:
        encoded_seq = seq
        if len(seq) > MAX_SEQ_LEN:
            encoded_seq = encoded_seq[-1*MAX_SEQ_LEN:]
            
        #padded_seq = pad_sequences([encoded_seq], maxlen=MAX_SEQ_LEN, padding='pre')
        padded_seq = np.array([seq])
        tags = to_categorical(np.full((1, padded_seq[0].shape[0]), tag), TAG_SIZE)
        y_prob = model.predict([padded_seq,tags])[0][-1].reshape(1,-1)#[3:].reshape(-1,1)
        
        if random.random() <= exploration_factor:
            probabilistic = True
        else:
            probabilistic = False
            
        if probabilistic:
            y_class = np.argmax(np.random.multinomial(1,y_prob[0]/(np.sum(y_prob[0])+1e-5),1))
        else:
            y_class = y_prob.argmax(axis=-1)[0]
        
        if y_class == 0:
            break
        out_word = reverse_word_map[y_class]
        seq.append(y_class)
        if out_word == 'eos' or len(seq) > maxlen or out_word == 'sos':
            break
    
    words = [reverse_word_map[idx] for idx in seq]
    
    return ' '.join(words)

In [ ]:
K.clear_session()
sess = tf.Session()
K.set_session(sess)
model = load_model('models/jokes_controlled_stacked_lstm_glove_gen1.hdf5')
with open('models/jokes_controlled_stacked_lstm_glove_tokenizer_1.pickle', 'rb') as pickleFile:
    tokenizer = pickle.load(pickleFile)

In [ ]:
joke = generate_categorical(model, tokenizer, "sos i had to use", maxlen=40, tag=0)
print(joke)

In [ ]:
def bigrams_list(sentence):
    words = sentence.split(' ')
    bigrams = []
    for i in range(0, len(words)-1):
        bigrams.append(words[i]+' '+words[i+1])
    return bigrams

print(bigrams_list("sos hello , i'm a dinosaur . eos"))

In [ ]:
sentence_bigrams = [bigrams_list(s) for s in sentences]
print(sentence_bigrams[:2])

In [ ]:
def intersection(lst1, lst2):
    temp = set(lst2)
    lst3 = [value for value in lst1 if value in temp]
    return lst3

def similarity_score(lst1, lst2):
    intersection_len = len(intersection(lst1, lst2))
    return (1.0*intersection_len)/len(lst1)#+len(lst2)-intersection_len)
 
def print_closest_sentences(sentence, sentence_bigrams, top_k=3):
    bigrams = bigrams_list(sentence)
    scores = np.array([similarity_score(bigrams, sbigrams)
                       for sbigrams in sentence_bigrams])
    top_k_indices = scores.argsort()[-1*top_k:][::-1]
    top_k_scores = scores[top_k_indices]
    for k in range(top_k):
        print(top_k_scores[k], " -> ", sentences[top_k_indices[k]])

In [ ]:
print_closest_sentences(joke, sentence_bigrams, 10)

In [ ]:
joke = generate_categorical(model, tokenizer, "eos bastard racist you", maxlen=40, tag=1)
joke = ' '.join(joke.split(' ')[::-1])
print(joke)
print_closest_sentences(joke, sentence_bigrams, 10)

In [ ]:
joke = generate_categorical(model, tokenizer, "eos . sport contact", maxlen=40, tag=1)
joke = ' '.join(joke.split(' ')[::-1])
print(joke)
print_closest_sentences(joke, sentence_bigrams, 10)

In [ ]:
joke = generate_categorical(model, tokenizer, "sos what do you call", maxlen=40, tag=0)
print(joke)
print_closest_sentences(joke, sentence_bigrams, 10)

In [ ]:
#Evaluation Model
#S_sem+S_wo

In [ ]:
import nltk

In [ ]:
from nltk.corpus import wordnet as wn
from nltk.stem import *

In [ ]:
nltk.download('wordnet')

In [ ]:
a = wn.synset('walk.v.01')
b = wn.synset('run.v.01')
a.wup_similarity(b)

In [ ]:
def wup_sim_score(word,tag,lst):
    score = 0
    synsets = wordnet.synset()
    for word2 in lst:
        score = max(score,nltk)

In [ ]:
def semantic_sim(sent1,sent2):
    sent1_words = nltk.word_tokenize(sent1)
    sen2_words = nltk.word_tokenize(sent2)
    stemmer = nltk.stem.PorterStemmer()
    for i in range(len(sent1_words)):
        sent1_words[i] = stemmer.stem(sent1_words[i])
    for i in range(len(sent2_words)):
        sent2_words[i] = stemmer.stem(sent2_words[i])
    sent1_pos_tags = nltk.pos_tag(sent1_words)
    sent2_pos_tags = nltk.pos_tag(sent2_words)
    sent1_pos_dict = {}
    sent2_pos_dict = {}
    
    for sent in sent1_pos_tags:
        try:
            sent1_pos_dict[sent[1]].append(sent[0])
        except:
            sent1_pos_dict[sent[1]] = []
            sent1_pos_dict[sent[1]].append(sent[0])
            
    for sent in sent2_pos_tags:
        try:
            sent2_pos_dict[sent[1]].append(sent[0])
        except:
            sent2_pos_dict[sent[1]] = []
            sent2_pos_dict[sent[1]].append(sent[0])
            
    term1 = 0
    for word,tag in sent1_pos_tags:
        term1 += wup_sim_score(word,sent2_pos_dict[tag])
        
    term2 = 0
    for word,tag in sent2_pos_tags:
        term2 += wup_sim_score(word,sent1_pos_dict[tag])
        
    return (term1+term2)/(len(sent1_words)+len(sent2_words))
    

In [ ]:
#Phrase overlap similarity

In [ ]:
import numpy as np

In [ ]:
def unigram_list(sentence):
    return sentence.split(' ')
def bigrams_list(sentence):
    words = sentence.split(' ')
    bigrams = []
    for i in range(0, len(words)-1):
        bigrams.append(words[i]+' '+words[i+1])
    return bigrams

print(bigrams_list("sos hello , i'm a dinosaur . eos"))
def trigrams_list(sentence):
    words = sentence.split(' ')
    trigrams = []
    for i in range(0, len(words)-2):
        trigrams.append(words[i]+' '+words[i+1]+' '+words[i+2])
    return trigrams

print(trigrams_list("sos hello , i'm a dinosaur . eos"))
def fourgrams_list(sentence):
    words = sentence.split(' ')
    fourgrams = []
    for i in range(0, len(words)-3):
        fourgrams.append(words[i]+' '+words[i+1]+' '+words[i+2]+words[i+3])
    return fourgrams

print(fourgrams_list("sos hello , i'm a dinosaur . eos"))
def phraseOverlap(sent1,sent2,flist):
    score = 0
    for i in range(0,len(flist),1):
        l1 = flist[i](sent1)
        l2 = set(flist[i](sent2))
        for l in l1:
            if l in l2:
                score += (i+1)**2
        print(score)
                
    return float(score)/(len(flist[0](sent1))+len(flist[0](sent2)))

In [ ]:
flist = [unigram_list,bigrams_list,trigrams_list,fourgrams_list]

In [ ]:
np.tanh(phraseOverlap("this is good stuff","is this good stuff",flist))